In [1]:
import numpy as np
import time
import cv2
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from pathlib import Path

# Практические задания

#### 1. Своё matmul. Реализуйте умножение матриц без @ и np.dot, через три вложенных цикла. Сравните скорость с NumPy на матрицах 200×200.

In [2]:
# Создаем функцию для умножения матриц
def my_matmul(A, B):
    # Делаем проверку на совместимость матриц, иначе выводим ошибку
    if len(A[0]) != len(B):
        raise ValueError('Несовместимые размеры')

    m = len(A)  # Число строк матрицы A
    n = len(B[0])  # Число столбцов матрицы B
    p = len(A[0])  # Число столбцов A (и одновременно число строк B)

    # Создаем матрицу mxn и заполняем ее нулями
    C = np.zeros((m, n))
    # Используем 3 вложенных цикла
    for i in range(m):
        for j in range(n):
            for k in range(p):
                C[i][j] += A[i][k] * B[k][j]  # Выполняем матричное умножение
    return C  # Возращаем результат

# Делаем проверку должно получится [19, 22] [43, 50]
A = [[1, 2], [3, 4]]  
B = [[5, 6], [7, 8]]
C = my_matmul(A, B)
print(C)

[[19. 22.]
 [43. 50.]]


In [3]:
# Генерация случайных матриц 200x200 для теста
X = np.random.rand(200, 200)
Y = np.random.rand(200, 200)

# Измеряем скорость нашей функции
start = time.time()
C1 = my_matmul(X, Y)
t_my = time.time() - start

# Измеряем скорость NumPy
start = time.time()
C2 = np.dot(X, Y)
t_np = time.time() - start

print(f"Моя функция: {t_my:.4f} сек")
print(f"NumPy:        {t_np:.4f} сек")
print(f"Ускорение:    {t_my / t_np:.1f}x")

# Проверка точности
print("Разница:", np.allclose(C1, C2))

Моя функция: 14.7331 сек
NumPy:        0.0177 сек
Ускорение:    831.9x
Разница: True


#### 2. Векторизация. Перепишите своё matmul через NumPy broadcasting без явных циклов. Сколько раз быстрее?

In [4]:
# Создаем функцию для переумножения матриц
def vectorized_matmul(A, B):
    # Преобразуем в массивы NumPy
    A = np.array(A)
    B = np.array(B)

    # Проверка совместимости размеров
    if A.shape[1] != B.shape[0]:
        raise ValueError('Несовместимые размеры: Число столбцов А не совпадает с числом строк В')

    # Делаем расширение размерностей
    A_exp = A[:, :, np.newaxis]  # Расширяем A форма (m, p, 1)
    B_exp = B[np.newaxis, :, :]  # Расширяем B форма (1, p, n)
    
    # Выполняем поэлементное умножение с broadcasting
    product = A_exp * B_exp
    
    # Выполняем суммирование по промежуточной оси k(axis=1)
    C = np.sum(product, axis=1)
    return C

# Делаем проверку
A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]
С = vectorized_matmul(A, B)
print(С)

[[19 22]
 [43 50]]


In [5]:
# Генерация случайных матриц 200x200 для теста
X = np.random.rand(200, 200)
Y = np.random.rand(200, 200)

# Измеряем скорость цикличной функции my_matmul
start = time.perf_counter()
C_my = my_matmul(X, Y)
t_my = time.perf_counter() - start

# Измеряем скорость векторизованной функции
start = time.perf_counter()
C_vec = vectorized_matmul(X, Y)
t_v = time.perf_counter() - start

# Измеряем скорость NumPy np.dot
start = time.time()
C_np = np.dot(X, Y)
t_np = time.time() - start

print(f"Моя функция:  {t_my:.4f} сек")
print(f"Век. функция: {t_v:.4f} сек")
print(f"NumPy:        {t_np:.4f} сек")
print(f"Ускорение (век. vs циклы):    {t_my / t_v:.1f}x")
print(f"Ускорение (NumPy vs век.):    {t_v / t_np:.1f}x")

# Проверка точности
assert np.allclose(C_my, C_vec)
assert np.allclose(C_vec, C_np)

Моя функция:  14.6326 сек
Век. функция: 0.0275 сек
NumPy:        0.0006 сек
Ускорение (век. vs циклы):    532.0x
Ускорение (NumPy vs век.):    48.8x


#### 3. Поворот картинки. Загрузите изображение, представьте каждый пиксель как точку (x, y), примените матрицу поворота на 30°. Сохраните результат.

In [6]:
def rotate_image(image, angle_degrees=30):
    # Загрузка и подготовка изображения
    if isinstance(image, str):
        img_array = cv2.imread(image)
    else:
        img_array = np.array(image).astype(np.uint8)
    
    # Приведим к RGB, если изображение серое
    if len(img_array.shape) == 2:
        img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
    
    height_src, width_src, _ = img_array.shape

    # Создаем матрицу поворота
    angle_rad = np.radians(angle_degrees)
    cos_theta = np.cos(angle_rad)
    sin_theta = np.sin(angle_rad)
    R = np.array([[cos_theta, -sin_theta],
                  [sin_theta, cos_theta]])

    # Расчёт новых размеров (чтобы вместить всё изображение)
    new_width = int(np.ceil(width_src * abs(cos_theta) + height_src * abs(sin_theta)))
    new_height = int(np.ceil(width_src * abs(sin_theta) + height_src * abs(cos_theta)))
    
    # Создание нового изображения
    rotated_img = np.zeros((new_height, new_width, 3), dtype=img_array.dtype)
    
    # Центр нового изображения
    center_x_new = new_width / 2
    center_y_new = new_height / 2

    # Обратное преобразование (для каждого пикселя нового изображения)
    for y_new in range(new_height):
        for x_new in range(new_width):
            # Смещение к центру нового изображения
            x_centered = x_new - center_x_new
            y_centered = y_new - center_y_new
            
            # Обратное преобразование (поворот на -angle_degrees)
            x_old = cos_theta * x_centered + sin_theta * y_centered + width_src / 2
            y_old = -sin_theta * x_centered + cos_theta * y_centered + height_src / 2
            
            # Билинейная интерполяция
            if 0 <= x_old < width_src - 1 and 0 <= y_old < height_src - 1:
                x0, y0 = int(x_old), int(y_old)
                dx, dy = x_old - x0, y_old - y0
                
                # Взвешивание соседних пикселей
                color = (1 - dx) * (1 - dy) * img_array[y0, x0] \
                      + dx * (1 - dy) * img_array[y0, x0 + 1] \
                      + (1 - dx) * dy * img_array[y0 + 1, x0] \
                      + dx * dy * img_array[y0 + 1, x0 + 1]
                
                rotated_img[y_new, x_new] = np.clip(color, 0, 255).astype(np.uint8)
    
    return rotated_img

# Создаём тестовое изображение 4x3
test_img = np.array([
    [[255, 0, 0], [0, 255, 0], [0, 0, 255]],   # Красный, Зелёный, Синий
    [[255, 255, 0], [0, 255, 255], [255, 0, 255]],  # Жёлтый, Голубой, Фиолетовый
    [[0, 0, 0], [255, 255, 255], [128, 128, 128]],  # Чёрный, Белый, Серый
    [[100, 150, 200], [50, 100, 150], [200, 150, 100]]  # Произвольные цвета
], dtype=np.uint8)

# Поворот на 30 градусов
rotated = rotate_image(test_img, 30)
# Проверка размеров
assert rotated.shape[0] > 3 and rotated.shape[1] > 4, "Неверные размеры после поворота"
# Проверка отсутствия чёрных пикселей (примерная)
non_black_pixels = np.sum(rotated > 10)
assert non_black_pixels > 10, "Изображение слишком много чёрных пикселей"
print(rotated)

[[[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  [  0   0   0]
  [  0   0   0]]

 [[  0   0   0]
  [  0   0   0]
  [174 246  76]
  [ 21 208 140]
  [  0   0   0]]

 [[  0   0   0]
  [  0   0   0]
  [178 216 208]
  [172 108 227]
  [  0   0   0]]

 [[  0   0   0]
  [  0   0   0]
  [134 147 159]
  [  0   0   0]
  [  0   0   0]]

 [[  0   0   0]
  [  0   0   0]
  [  0   0   0]
  [  0   0   0]
  [  0   0   0]]]


#### 4. Линейная регрессия с нуля. На датасете Boston Housing решите X^T X w = X^T y. Сравните MSE с sklearn.LinearRegression.

In [3]:
# Создаем функцию
def linear_regression_from_scratch(X, y):
    # Создаем вектор из единиц длиной m (количество строк в X)
    ones = np.ones((X.shape[0], 1))
    # Объединяем вектор единиц и исходную матрицу X по горизонтали
    X_with_bias = np.hstack([ones, X])
    # Транспонирование + матричное умножение
    w = np.linalg.lstsq(X_with_bias, y, rcond=None)[0]
    return w

# Загрузка данных
current_dir = Path.cwd()
root_dir = current_dir.parent.parent
data_path = root_dir / "data" / "BostonHousing.csv"
df = pd.read_csv(data_path)
target_col = "medv"
X = df.drop(columns=[target_col]).values
y = df[target_col].values

# Разделение
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Удаление столбцов с NaN
nan_mask = np.any(np.isnan(X_train), axis=0)
X_train = X_train[:, ~nan_mask]
X_test = X_test[:, ~nan_mask]

# Масштабирование
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Обучение модели
w_my = linear_regression_from_scratch(X_train_scaled, y_train)
ones_test = np.ones((X_test_scaled.shape[0], 1))
X_test_with_bias = np.hstack([ones_test, X_test_scaled])
y_pred_my = X_test_with_bias @ w_my

mse_my = np.mean((y_test - y_pred_my) ** 2)
rmse_my = np.sqrt(mse_my)
print(f"MSE на тесте: {mse_my:.2f}")
print(f"RMSE на тесте: {rmse_my:.2f}")

model = LinearRegression()
model.fit(X_train_scaled, y_train)  # Используем масштабированные данные
y_pred_sk = model.predict(X_test_scaled)
mse_sk = mean_squared_error(y_test, y_pred_sk)
print(f"MSE (мой метод): {mse_my:.2f}")
print(f"MSE (sklearn): {mse_sk:.2f}")

# Сравнение весов w_my с model.coef_ и model.intercept_
print("Мой свободный член:", w_my[0])
print("Sklearn intercept:", model.intercept_)
print("Мои коэффициенты:", w_my[1:])
print("Sklearn coef:", model.coef_)

MSE на тесте: 22.13
RMSE на тесте: 4.70
MSE (мой метод): 22.13
MSE (sklearn): 22.13
Мой свободный член: 22.796534653465358
Sklearn intercept: 22.796534653465343
Мои коэффициенты: [-1.11758917  1.11505834 -0.31605191  0.82337471 -2.39103148  0.62755408
 -3.65287983  3.37143708 -2.3203188  -2.56185653  0.86281191 -5.62747224]
Sklearn coef: [-1.11758917  1.11505834 -0.31605191  0.82337471 -2.39103148  0.62755408
 -3.65287983  3.37143708 -2.3203188  -2.56185653  0.86281191 -5.62747224]


#### 5. Псевдообратная. Реализуйте pinv(A) через SVD (потом поймёте, как). Проверьте, что A @ pinv(A) @ A ≈ A.

In [12]:
# Создаем функцию
def my_pinv(A):
    # Делаем вычисление SVD
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    
    # Порог для определения «нулевых» сингулярных значений
    if s.size == 0:
        return np.zeros((A.shape[1], A.shape[0]))
    tolerance = max(A.shape) * np.finfo(float).eps * max(s)
    # Векторизованное обращение ненулевых сингулярных значений
    s_inv = np.where(s > tolerance, 1.0 / s, 0.0)

    # Диагональная матрица из обращённых сингулярных значений
    S_inv = np.diag(s_inv)

    # Сборка псевдообратной матрицы
    A_pinv = Vt.T @ S_inv @ U.T
    return A_pinv

# Проверка свойства
A = np.array([[5, 3], [2, 7], [1, 4]], dtype=float)
A_pinv = my_pinv(A)
reconstructed = A @ A_pinv @ A
assert np.linalg.norm(A - reconstructed) < 1e-10, "Псевдообратная матрица не удовлетворяет свойству"
assert np.allclose(A_pinv, np.linalg.pinv(A))
print("Проверка пройдена: свойство A ≈ A @ A+ @ A выполняется.")

Проверка пройдена: свойство A ≈ A @ A+ @ A выполняется.


#### 6. Композиция преобразований. Сделайте поворот на 30°, потом масштабирование в 2 раза по X. Найдите итоговую матрицу одним умножением. Проверьте, что результат совпадает.

#### 7. Проверка вырожденности. Сгенерируйте 10 случайных матриц 5×5, добавьте к двум из них линейно зависимую строку. Найдите эти две через matrix_rank.

#### 8. Свой solve. Реализуйте решение Ax = b методом Гаусса (прямой и обратный ход). Сравните с np.linalg.solve.